# 02 · Capability registry and dispatch

**Role of this layer.** The registry is the *approved menu*: 12 pre-written,
tested capability functions, each described as data (inputs, parameters with
types and ranges, assumptions, minimum observations, library calls).

* The planner LLM is shown a summary **generated** from the registry, so it can
  never drift from what is enforced.
* The executor runs a step by looking its capability name up in the `REGISTRY`
  dict and calling the function attached to it. A name that is not a key has no
  function, so it cannot run: the whitelist holds *by construction*, with no
  `eval`, `exec` or `getattr` anywhere.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.registry import REGISTRY, llm_summary, registry_version

pd.DataFrame(
    [
        {
            "capability": spec.name,
            "inputs": ", ".join(i.name for i in spec.inputs) or "-",
            "params": ", ".join(p.name for p in spec.params) or "-",
            "output": spec.output_type.value,
            "min obs": spec.min_obs.description,
        }
        for spec in REGISTRY.values()
    ]
)

,capability,inputs,params,output,min obs
0,load_series,-,dataset_id,series,at least 1 observation
1,resample,series,"freq, how",series,at least 2 observations
2,compute_returns,series,"method, periods",series,at least 2 x periods observations
3,difference,series,periods,series,at least 2 x periods observations
4,spread,"a, b",-,series,at least 2 common observations
5,rolling_volatility,series,"window, annualize",series,at least 3 x window observations
6,flag_regime,series,"quantile, method, min_periods",flags,at least 2 x min_periods observations
7,forward_return,returns,horizon,series,at least 2 x horizon observations
8,summary_stats,series,-,table,at least 4 observations
9,compare_groups,"values, flags","test, nonoverlapping",test_result,at least 20 observations in each group (flagged and normal)


One spec in full: parameters with ranges, assumptions, and the library call behind it.

In [3]:
spec = REGISTRY["rolling_volatility"]
print(spec.description)
print("library calls:", spec.library_calls)
for p in spec.params:
    print(
        f"  param {p.name}: {p.type}, required={p.required}, default={p.default}, "
        f"range=[{p.min}, {p.max}]"
    )
for a in spec.assumptions:
    print(f"  assumption ({'needs approval' if a.requires_approval else 'info'}): {a.text}")

Trailing rolling volatility (sample standard deviation) of returns, changes or a spread, annualised by default.
library calls: pandas.Series.rolling(window).std(ddof=1) * sqrt(factor)
  param window: int, required=True, default=None, range=[4, 260]
  param annualize: bool, required=False, default=True, range=[None, None]
  assumption (info): Annualised by sqrt(periods per year): 252 daily, 52 weekly, 12 monthly (divided by k for k-period returns or changes).


The LLM-facing summary is JSON generated from these specs (no functions, no
library calls). Its SHA-256 is the **registry version**, stored in every run
record and in the plan-cache key.

In [4]:
print(llm_summary()[:900], "...")
print("registry version:", registry_version())

[
 {
  "name": "load_series",
  "description": "Load one dataset from the catalog as a daily time series (hash-verified).",
  "inputs": [],
  "params": [
   {
    "name": "dataset_id",
    "type": "str",
    "required": true,
    "description": "Id of a dataset from the dataset list (e.g. NASDAQCOM)."
   }
  ],
  "output": {
   "data_type": "series",
   "kind": "from_catalog"
  },
  "min_obs": "at least 1 observation",
  "assumptions": [
   {
    "text": "Uses the committed FRED snapshot, verified by SHA-256; missing values (holidays) are dropped, not filled.",
    "requires_approval": false
   }
  ]
 },
 {
  "name": "resample",
  "description": "Convert a level series to weekly (week ending Friday) or monthly (month end) frequency. Resample levels BEFORE computing returns or changes.",
  "inputs": [
   {
    "name": "series",
    "data_types": [
     "series"
    ],
    "kinds": [
      ...
registry version: 3ad9cf225d152d1b34dc907c59941f0285513326da84ebdacb38dbeebced4302


### Capabilities are plain functions over Artifacts

Each function takes Artifacts (value + metadata) and returns a new Artifact with
its metadata set explicitly. Here: weekly NASDAQ returns and their 12-week
annualised volatility. Note how `kind`, `frequency` and `return_method`
propagate, and how the annualisation factor is recorded as an assumption.

In [5]:
from quantlab import capabilities as cap
from quantlab.data import load_catalog

catalog = load_catalog(settings.catalog_path)
prices = cap.load_series("NASDAQCOM", catalog=catalog)
weekly = cap.resample(prices, freq="W", how="last")
returns = cap.compute_returns(weekly, method="simple")
vol = cap.rolling_volatility(returns, window=12)
for name, artifact in [("prices", prices), ("weekly", weekly), ("returns", returns), ("vol", vol)]:
    m = artifact.meta
    print(
        f"{name:8} kind={m.kind} freq={m.frequency} method={m.return_method} "
        f"n={m.n_obs} {m.start}..{m.end} notes={m.notes}"
    )

prices   kind=price_level freq=D method=None n=14040 1971-02-05..2026-10-13 notes=['dropped 487 missing values out of 14527 rows (not filled: filling would invent prices on holidays)']
weekly   kind=price_level freq=W method=None n=2906 1971-02-05..2026-10-09 notes=['resampled to W (W-FRI) using the period last']
returns  kind=returns freq=W method=simple n=2905 1971-02-12..2026-10-09 notes=['simple returns over 1 period(s)']
vol      kind=volatility freq=W method=simple n=2894 1971-04-30..2026-10-09 notes=['sample std (ddof=1) over 12 observations, annualised by sqrt(52)']


### Dispatch refuses anything that is not registered

In [6]:
from quantlab.executor import CapabilityRefusedError, dispatch
from quantlab.plan import Step

for name in ["random_forest", "os.system", "eval"]:
    try:
        dispatch(Step(id="s1", capability=name), {}, catalog)
    except CapabilityRefusedError as error:
        print("refused:", error)

ok = dispatch(Step(id="s1", capability="load_series", params={"dataset_id": "VIXCLS"}), {}, catalog)
print("load_series ran:", ok.meta.n_obs, "observations of", ok.meta.source_dataset_ids)

refused: 'random_forest' is not a registered capability.
refused: 'os.system' is not a registered capability.
refused: 'eval' is not a registered capability.
load_series ran: 9294 observations of ['VIXCLS']
